# 🏭 AR Synthetic Data Pipeline

> **Notebook:** `data_generator.ipynb`  
> **Stage:** 1 of 2 — Data Generation Layer  
> **Output:** `data/raw/AR_Invoices_950K.csv` · `Customers_Master.csv` · `Bank_Documents_Tracking.csv`

This notebook demonstrates a **high-performance data engineering approach** to generating large-scale synthetic datasets (950K+ records) using vectorized operations with NumPy and Pandas, structured for a typical **Star Schema** (Fact and Dimension tables).

**What gets generated:**
1. `dim_Customers` — 10,000 customer records (company names + payment terms)
2. `fact_AR_Invoices` — 950,000 invoice records spanning a 2-year posting window
3. `fact_Bank_Documents` — ~399K bank submission records (70% of open/partial invoices)

---
## 📦 Dependencies

Install required libraries if not already present in the environment.

In [1]:
!pip install pandas numpy faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   -------------------- ------------------- 1.0/2.0 MB 2.9 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 3.0 MB/s eta 0:00:00


---
## ⚙️ Setup & Path Configuration

Import all required libraries and resolve project paths using `pathlib` — ensuring outputs are always saved to `data/raw/` regardless of the working directory.

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
from pathlib import Path
import time

# ── Resolve project root (works both in Jupyter and as a script) ──────────────
NOTEBOOK_DIR = Path.cwd()  # notebooks/ when run from Jupyter
PROJECT_ROOT = NOTEBOOK_DIR.parent

# ── Output directory — all CSVs go to data/raw/ ───────────────────────────────
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Environment ready')
print(f'   Project root : {PROJECT_ROOT}')
print(f'   Output dir   : {RAW_DATA_DIR}')

---
## ⏱️ Step 1 — Initialize Timer & Faker

We start an execution timer to benchmark the full pipeline and initialize the `Faker` instance used for generating realistic company names.

In [3]:
# ── Start execution timer ─────────────────────────────────────────────────────
start_time = time.time()

# ── Initialize Faker for dimension table generation ───────────────────────────
fake = Faker()

---
## 📐 Step 2 — Define Dataset Scale

All volume constants are declared here for easy tuning without modifying generation logic.

| Constant | Value | Description |
|----------|-------|-------------|
| `NUM_CUSTOMERS` | 10,000 | Unique customer entities in the dimension table |
| `NUM_INVOICES` | 950,000 | Total invoice records in the fact table |

In [4]:
# ── Dataset scale constants ───────────────────────────────────────────────────
NUM_CUSTOMERS = 10000
NUM_INVOICES  = 950000

print('Generating Customers Dimension...')

Generating Customers Dimension...


---
## 👥 Step 3 — Generate Customer Dimension Table

We use `Faker` to produce realistic company names for 10,000 customers.  
Payment terms are sampled from the standard set used in AR: **30 / 60 / 90 / 120 days**.

> Faker is used here (not NumPy) because this is a small dimension table — performance is not a concern at this scale.

In [5]:
# ── Dimension table: Customers (Faker used for realistic company names) ────────
customers_df = pd.DataFrame({
    'CustomerID'  : [f'C-{(i+1):05d}' for i in range(NUM_CUSTOMERS)],
    'CustomerName': [fake.company() for _ in range(NUM_CUSTOMERS)],
    'PaymentTerms': np.random.choice([30, 60, 90, 120], NUM_CUSTOMERS)
})

print('Generating Invoices Fact Table using NumPy (High Speed)...')

Generating Invoices Fact Table using NumPy (High Speed)...


---
## 📅 Step 4 — Prepare Vectorized Date Range

We generate random posting dates across a **730-day (2-year) window** using NumPy's vectorized integer sampling — avoiding slow Python-level date loops entirely.

In [6]:
# ── Vectorized date range setup (avoids slow Python date loops) ───────────────
end_date         = datetime.today()
start_date       = end_date - timedelta(days=730)
date_range_days  = (end_date - start_date).days

# ── Sample a random number of days to add for each invoice ───────────────────
random_days_to_add = np.random.randint(0, date_range_days, NUM_INVOICES)

---
## 🧾 Step 5 — Generate AR Invoices Fact Table

The core fact table is built with **fully vectorized NumPy operations** for maximum throughput at 950K records.

| Column | Distribution / Range |
|--------|---------------------|
| `InvoiceID` | Sequential: `INV-0000001` → `INV-0950000` |
| `CustomerID` | Random FK from `dim_Customers` |
| `PostingDate` | Uniform random over 730-day window |
| `Amount` | Uniform: $100 – $150,000 |
| `Status` | Open 40% · Partial 20% · Cleared 40% |

In [7]:
# ── Vectorized date generation ────────────────────────────────────────────────
posting_dates = start_date + pd.to_timedelta(random_days_to_add, unit='D')

# ── Fact table: AR Invoices (NumPy vectorized for maximum speed) ──────────────
invoices_df = pd.DataFrame({
    'InvoiceID' : [f'INV-{(i+1):07d}' for i in range(NUM_INVOICES)],
    'CustomerID': np.random.choice(customers_df['CustomerID'], NUM_INVOICES),
    'PostingDate': posting_dates,
    'Amount'    : np.round(np.random.uniform(100.0, 150000.0, NUM_INVOICES), 2),
    'Status'    : np.random.choice(['Open', 'Partial', 'Cleared'], NUM_INVOICES, p=[0.4, 0.2, 0.4])
})

print('Generating Bank 2nd Set Tracking Table...')

Generating Bank 2nd Set Tracking Table...


---
## 🏦 Step 6 — Generate Bank Documents Tracking Table

We simulate bank submission records for **70% of Open/Partial invoices** — reflecting a realistic scenario where most outstanding invoices are submitted to the bank for collection.

| Column | Distribution |
|--------|--------------|
| `BankSubmissionDate` | 5–30 days after the invoice posting date |
| `DocStatus` | Under Review 60% · Accepted 30% · Rejected 10% |

In [8]:
# ── Extract pending invoices (Open or Partial) that were submitted to the bank ─
pending_invoices = invoices_df[invoices_df['Status'].isin(['Open', 'Partial'])]['InvoiceID'].values

# ── Assume 70% of pending invoices were forwarded to the bank ─────────────────
NUM_BANK_DOCS = int(len(pending_invoices) * 0.7)

bank_docs_df = pd.DataFrame({
    'DocID'             : [f'DOC-{(i+1):06d}' for i in range(NUM_BANK_DOCS)],
    'InvoiceID'         : np.random.choice(pending_invoices, NUM_BANK_DOCS, replace=False),
    'BankSubmissionDate': posting_dates[:NUM_BANK_DOCS] + pd.to_timedelta(
                              np.random.randint(5, 30, NUM_BANK_DOCS), unit='D'
                          ),
    'DocStatus'         : np.random.choice(
                              ['Under Review', 'Accepted', 'Rejected'],
                              NUM_BANK_DOCS, p=[0.6, 0.3, 0.1]
                          )
})

---
## 💾 Step 7 — Export to CSV

All three tables are exported to `data/raw/` using `pathlib`-based paths — no hardcoded backslashes, works on any OS.

In [9]:
# ── Export all tables to data/raw/ ────────────────────────────────────────────
print('Exporting to CSV...')

customers_df.to_csv(RAW_DATA_DIR / 'Customers_Master.csv',        index=False)
invoices_df.to_csv( RAW_DATA_DIR / 'AR_Invoices_950K.csv',        index=False)
bank_docs_df.to_csv(RAW_DATA_DIR / 'Bank_Documents_Tracking.csv', index=False)

print(f'Done! Total execution time: {round(time.time() - start_time, 2)} seconds.')

Exporting to CSV...
Done! Total execution time: 10.76 seconds.


---
## ✅ Step 8 — Data Quality Assurance (DQA)

As a best practice in data engineering, we verify the **integrity of the generated data** before it is consumed by downstream processes (Power Query ETL, dashboards).

| Check | Expected |
|-------|----------|
| Invoices Count | 950,000 |
| Customers Count | 10,000 |
| Bank Documents Count | ~399,000 (70% of Open/Partial) |
| Referential Integrity | 0 orphan invoices |

In [10]:
# ── Data Quality Assurance ────────────────────────────────────────────────────
print('--- Data Summary ---')
print(f'Invoices Count: {len(invoices_df):,}')
print(f'Customers Count: {len(customers_df):,}')
print(f'Bank Documents Count: {len(bank_docs_df):,}')

# ── Referential integrity check: no invoices should have an unknown CustomerID ─
missing_customers = invoices_df[~invoices_df['CustomerID'].isin(customers_df['CustomerID'])]
print(f'Referential Integrity Issues: {len(missing_customers)}')

# ── Preview the fact table ────────────────────────────────────────────────────
display(invoices_df.head())
display(invoices_df.describe())

--- Data Summary ---
Invoices Count: 950,000
Customers Count: 10,000
Bank Documents Count: 399,219
Referential Integrity Issues: 0


,InvoiceID,CustomerID,PostingDate,Amount,Status
0,INV-0000001,C-01221,2024-10-05 18:42:03.727963,134033.68,Cleared
1,INV-0000002,C-07499,2026-03-06 18:42:03.727963,72333.33,Cleared
2,INV-0000003,C-02056,2024-12-16 18:42:03.727963,67937.34,Cleared
3,INV-0000004,C-05863,2024-12-24 18:42:03.727963,81729.98,Cleared
4,INV-0000005,C-04854,2024-06-21 18:42:03.727963,9035.96,Cleared


,PostingDate,Amount
count,950000,950000.000000
mean,2025-04-20 03:20:36.731752448,75001.579183
min,2024-04-20 18:42:03.727963,100.110000
25%,2024-10-19 18:42:03.727962880,37544.650000
50%,2025-04-19 18:42:03.727962880,74959.140000
75%,2025-10-19 18:42:03.727962880,112515.307500
max,2026-04-19 18:42:03.727963,149999.900000
std,NaN,43285.640499
